# Florian Ewing, Margaret Keau, John Farnandez, Lauren Connely
# Professor Ix Procopios
# Software Design and Implementation
# Adapted: 2014–2024 BTS Flight Delay Dataset

# Optimum Departure Location & Airline by Region Predictor
# (2014–2024 BTS Dataset Adaptation)

This notebook adapts the original experiment to the larger
**2014–2024 Delayed Flight Data** dataset from Kaggle
(`https://www.kaggle.com/datasets/ce31d28fde1e922e46342846c30171045802a67cb3356b3b4520607ecf569a12`).

That dataset follows the standard **BTS (Bureau of Transportation Statistics)**
on-time performance schema, which uses different column names from the
original `bordanova` dataset. This notebook handles all column-name
differences and maps everything back to the same modelling pipeline.

### Key schema differences vs. the original notebook
| Original column | BTS column |
|---|---|
| `Dep_Airport` | `ORIGIN` |
| `Arr_Airport` | `DEST` |
| `Airline` | `OP_UNIQUE_CARRIER` or `OP_CARRIER` |
| `Dep_Delay` | `DEP_DELAY` |
| `FlightDate` | `FL_DATE` |
| `Tail_Number` | `TAIL_NUM` |
| `Cancelled` | `CANCELLED` |
| `Aircraft_age` | **not in BTS** — dropped or sourced separately |
| `DepWeather_*` | **not in BTS** — dropped (weather merge optional) |

### What changes in the model
- `Aircraft_age`, `Dep_LAT`, `Dep_LON`, and all weather features are
  **not natively in the BTS dataset**. We drop them from `FINAL_FEATURES`
  by default. Re-add them if you join an external weather or aircraft-age
  source.
- The holiday list is extended from 2014 through 2024.
- Sampling is increased (or chunked) because the dataset is ~10× larger.

# Cell 1 — Install & Download

The dataset is downloaded via `kagglehub`. Make sure your Kaggle API
credentials are configured (either via `~/.kaggle/kaggle.json` or the
Colab secrets manager).

In [ ]:
# ===== Cell 1: Install & Download =====

# Uncomment if running in Colab / fresh environment
# !pip install -q kagglehub

import os
import pandas as pd
import numpy as np
import kagglehub

# ------------------------------------------------------------------
# Download the 2014-2024 BTS delayed-flight dataset.
# The hash-based URL resolves to the dataset at:
# https://www.kaggle.com/datasets/ce31d28fde1e922e46342846c30171045802a67cb3356b3b4520607ecf569a12
# ------------------------------------------------------------------
DATASET_SLUG = "ce31d28fde1e922e46342846c30171045802a67cb3356b3b4520607ecf569a12"

dataset_path = kagglehub.dataset_download(DATASET_SLUG)
print("Downloaded to:", dataset_path)

# List files so we know what we're working with
for f in sorted(os.listdir(dataset_path)):
    size_mb = os.path.getsize(os.path.join(dataset_path, f)) / 1e6
    print(f"  {f:50s}  {size_mb:8.1f} MB")

# Cell 2 — Load & Normalise Columns

The BTS dataset is typically delivered as one CSV per year (or one large
combined file). We load everything and immediately rename columns to the
internal names used throughout this notebook, so the rest of the pipeline
is identical to the original.

In [ ]:
# ===== Cell 2: Load & Normalise Columns =====

# ------------------------------------------------------------------
# BTS → internal column mapping
# ------------------------------------------------------------------
BTS_RENAME = {
    # Date
    "FL_DATE":            "FlightDate",
    # Carrier  (BTS uses IATA 2-letter codes; we keep them as-is)
    "OP_UNIQUE_CARRIER":  "Airline",
    "OP_CARRIER":         "Airline",          # fallback if UNIQUE absent
    # Airports
    "ORIGIN":             "Dep_Airport",
    "DEST":               "Arr_Airport",
    # Delay
    "DEP_DELAY":          "Dep_Delay",
    "ARR_DELAY":          "Arr_Delay",
    # Cancellation
    "CANCELLED":          "Cancelled",
    # Tail number (for future joins)
    "TAIL_NUM":           "Tail_Number",
    # Distance
    "DISTANCE":           "Distance",
    # Departure hour (useful temporal feature)
    "CRS_DEP_TIME":       "Sched_Dep_Time",
}

# ------------------------------------------------------------------
# Columns we actually need (saves memory on a ~60 M row dataset)
# ------------------------------------------------------------------
KEEP_RAW = [
    "FL_DATE",
    "OP_UNIQUE_CARRIER",  "OP_CARRIER",   # one will be present
    "ORIGIN",            "DEST",
    "DEP_DELAY",         "ARR_DELAY",
    "CANCELLED",
    "TAIL_NUM",
    "DISTANCE",
    "CRS_DEP_TIME",
]

def load_bts_file(path, keep=KEEP_RAW, rename=BTS_RENAME):
    """Load a single BTS CSV, keeping only relevant columns."""
    # Peek at header to know which columns are present
    header = pd.read_csv(path, nrows=0).columns.tolist()
    cols_to_read = [c for c in keep if c in header]
    df = pd.read_csv(
        path,
        usecols=cols_to_read,
        low_memory=False,
    )
    # Rename
    df.rename(columns={k: v for k, v in rename.items() if k in df.columns},
              inplace=True)
    # If both OP_UNIQUE_CARRIER and OP_CARRIER survived renaming, drop the dup
    if "Airline" not in df.columns:
        raise KeyError("Could not find a carrier column in: " + path)
    # Parse date
    df["FlightDate"] = pd.to_datetime(df["FlightDate"])
    return df

# ------------------------------------------------------------------
# Load all CSV files in the dataset directory
# (handles both single-file and multi-file layouts)
# ------------------------------------------------------------------
csv_files = sorted([
    os.path.join(dataset_path, f)
    for f in os.listdir(dataset_path)
    if f.endswith(".csv")
])

print(f"Found {len(csv_files)} CSV file(s). Loading…")
chunks = []
for fp in csv_files:
    try:
        df_chunk = load_bts_file(fp)
        chunks.append(df_chunk)
        print(f"  Loaded {fp.split('/')[-1]:50s}  rows={len(df_chunk):,}")
    except Exception as e:
        print(f"  SKIP {fp.split('/')[-1]} — {e}")

flights = pd.concat(chunks, ignore_index=True)
print(f"\nTotal rows loaded: {len(flights):,}")
print(flights.dtypes)

# Cell 3 — Create Target Variables & Class Weights

Same logic as the original notebook:
- **Delay_Class 0** → departed early or within 14 minutes of schedule
- **Delay_Class 1** → departed 15 or more minutes late

Cancellations are flagged separately.

In [ ]:
# ===== Cell 3: Target Variables =====

# Drop rows without a departure delay (diverted / cancelled have NaN)
flights = flights.dropna(subset=["Dep_Delay"])

# Binary delay target  (≥ 15 min = delayed)
flights["Delay_Class"] = np.where(flights["Dep_Delay"] >= 15, 1, 0)

# Cancellation flag (BTS uses 1.0 / 0.0 already)
flights["Cancelled_Class"] = flights["Cancelled"].fillna(0).astype(int)

# Class distribution
delay_counts   = flights["Delay_Class"].value_counts()
delay_pct      = flights["Delay_Class"].value_counts(normalize=True) * 100
cancel_counts  = flights["Cancelled_Class"].value_counts()
cancel_pct     = flights["Cancelled_Class"].value_counts(normalize=True) * 100

print("Delay Class Counts:");    print(delay_counts)
print("\nDelay Class %:");        print(delay_pct.round(2))
print("\nCancellation Counts:"); print(cancel_counts)
print("\nCancellation %:");      print(cancel_pct.round(2))

# Inverse-frequency class weights
total   = len(flights)
weight_0 = total / (2 * delay_counts[0])
weight_1 = total / (2 * delay_counts[1])
class_weights = {0: weight_0, 1: weight_1}
print("\nSuggested class weights:", class_weights)

# Cell 4 — Region Mapping

Identical to the original. The BTS `ORIGIN` column already contains
3-letter IATA codes, which map directly onto the airport→state→region
lookup used before.

Because the BTS dataset does **not** include a geolocation file, we fetch
state information from the BTS `ORIGIN_STATE_ABR` column (if present) or
fall back to a small bundled lookup table.

In [ ]:
# ===== Cell 4: Region Mapping =====

region_to_states = {
    "Northeast": ["ME","NH","VT","MA","RI","CT","NY","NJ","PA","MD","DE"],
    "Southeast": ["VA","WV","KY","TN","NC","SC","GA","FL","AL","MS","AR","LA"],
    "Midwest":   ["OH","IN","IL","MI","WI","MN","IA","MO","ND","SD","NE","KS"],
    "Southwest": ["TX","OK","NM","AZ"],
    "West":      ["CO","WY","MT","ID","UT","NV","CA","OR","WA","AK","HI"],
}
state_to_region = {
    state: region
    for region, states in region_to_states.items()
    for state in states
}

# ------------------------------------------------------------------
# Build airport → region from BTS ORIGIN_STATE_ABR (if available)
# OR from a bundled IATA → state reference table
# ------------------------------------------------------------------
if "ORIGIN_STATE_ABR" in flights.columns:
    # Best case: state is already in the dataset
    airport_state = (
        flights[["Dep_Airport", "ORIGIN_STATE_ABR"]]
        .drop_duplicates()
        .rename(columns={"ORIGIN_STATE_ABR": "STATE"})
    )
else:
    # Fallback: download the BTS airport reference table
    # (tiny CSV, publicly hosted by BTS)
    print("ORIGIN_STATE_ABR not found — fetching BTS airport reference…")
    airport_ref_url = (
        "https://raw.githubusercontent.com/datasets/"
        "airport-codes/master/data/airport-codes.csv"
    )
    ref = pd.read_csv(airport_ref_url)
    # Keep US airports and extract state from iso_region (e.g. 'US-CA' → 'CA')
    ref = ref[ref["iso_country"] == "US"].copy()
    ref["STATE"] = ref["iso_region"].str.split("-").str[1]
    ref = ref.dropna(subset=["iata_code", "STATE"])
    airport_state = ref[["iata_code", "STATE"]].rename(
        columns={"iata_code": "Dep_Airport"}
    ).drop_duplicates()

airport_state["Region"] = airport_state["STATE"].map(state_to_region)
airports_region = airport_state[["Dep_Airport", "Region"]].dropna()

print(airports_region["Region"].value_counts())

# Cell 5 — Feature Engineering

Because the BTS dataset spans **2014–2024**, the holiday list is expanded
to cover all years. We also add a **scheduled departure hour** feature
(`CRS_DEP_TIME` → `Dep_Hour`) which is available in BTS data and is a
strong delay predictor.

Features **removed** vs. the original (not in BTS):
- `Aircraft_age`
- `Dep_LAT` / `Dep_LON`
- All `DepWeather_*` columns

These can be re-added by joining an external geolocation or weather file
using the same `merge_weather` / `airports_geo` pattern from the original.

In [ ]:
# ===== Cell 5: Feature Engineering =====
from sklearn.preprocessing import LabelEncoder

# ------------------------------------------------------------------
# Holiday proximity flag — extended to 2014-2024
# (New Year's Day, MLK, Presidents Day, Memorial Day, Independence Day,
#  Labor Day, Thanksgiving, Christmas for every year)
# ------------------------------------------------------------------
holiday_dates = []
for year in range(2014, 2025):
    holiday_dates += [
        f"{year}-01-01",  # New Year's Day
        f"{year}-07-04",  # Independence Day
        f"{year}-12-25",  # Christmas
    ]
# Approximate floating holidays (close enough for a 2-day window)
# MLK (3rd Mon Jan), Presidents (3rd Mon Feb), Memorial (last Mon May),
# Labor (1st Mon Sep), Thanksgiving (4th Thu Nov)
approx_floating = [
    "2014-01-20","2014-02-17","2014-05-26","2014-09-01","2014-11-27",
    "2015-01-19","2015-02-16","2015-05-25","2015-09-07","2015-11-26",
    "2016-01-18","2016-02-15","2016-05-30","2016-09-05","2016-11-24",
    "2017-01-16","2017-02-20","2017-05-29","2017-09-04","2017-11-23",
    "2018-01-15","2018-02-19","2018-05-28","2018-09-03","2018-11-22",
    "2019-01-21","2019-02-18","2019-05-27","2019-09-02","2019-11-28",
    "2020-01-20","2020-02-17","2020-05-25","2020-09-07","2020-11-26",
    "2021-01-18","2021-02-15","2021-05-31","2021-09-06","2021-11-25",
    "2022-01-17","2022-02-21","2022-05-30","2022-09-05","2022-11-24",
    "2023-01-16","2023-02-20","2023-05-29","2023-09-04","2023-11-23",
    "2024-01-15","2024-02-19","2024-05-27","2024-09-02","2024-11-28",
]
us_holidays = pd.to_datetime(holiday_dates + approx_floating)

def is_near_holiday(date_series, holidays, window=2):
    flags = np.zeros(len(date_series), dtype=int)
    for h in holidays:
        diff = (date_series - h).dt.days.abs()
        flags |= (diff <= window).astype(int).values
    return flags

# ---- Temporal features ----
flights["Month"]     = flights["FlightDate"].dt.month
flights["DayOfWeek"] = flights["FlightDate"].dt.dayofweek  # 0=Mon, 6=Sun
flights["IsWeekend"] = (flights["DayOfWeek"] >= 5).astype(int)
flights["IsHoliday"] = is_near_holiday(flights["FlightDate"], us_holidays)
flights["Year"]      = flights["FlightDate"].dt.year   # useful over 10-yr span

# ---- Scheduled departure hour (0–23) ----
# CRS_DEP_TIME is HHMM integer, e.g. 830 → 8:30 → hour 8
if "Sched_Dep_Time" in flights.columns:
    flights["Dep_Hour"] = (flights["Sched_Dep_Time"] // 100).clip(0, 23)
else:
    flights["Dep_Hour"] = np.nan

# ---- Merge region ----
region_map = airports_region.set_index("Dep_Airport")["Region"]
flights["Region"] = flights["Dep_Airport"].map(region_map)

# ---- Feature set (no post-departure leakage) ----
# Weather and geo columns are commented out — uncomment if you join them
FEATURE_COLS = [
    "Airline",
    "Dep_Airport",
    "Year",           # new: 10-yr span means year matters
    "Month",
    "DayOfWeek",
    "IsWeekend",
    "IsHoliday",
    "Dep_Hour",       # new: strong delay signal from BTS
    # "Aircraft_age",  # not in BTS natively
    # "Dep_LAT",       # not in BTS natively
    # "Dep_LON",       # not in BTS natively
    # "DepWeather_tavg",
    # "DepWeather_prcp",
    # "DepWeather_wspd",
]

model_df = flights[FEATURE_COLS + ["Delay_Class", "Region"]].dropna()

# ---- Encode categoricals ----
le_airline  = LabelEncoder()
le_airport  = LabelEncoder()

model_df = model_df.copy()
model_df["Airline_enc"]     = le_airline.fit_transform(model_df["Airline"])
model_df["Dep_Airport_enc"] = le_airport.fit_transform(model_df["Dep_Airport"])

FINAL_FEATURES = [
    "Airline_enc",
    "Dep_Airport_enc",
    "Year",
    "Month",
    "DayOfWeek",
    "IsWeekend",
    "IsHoliday",
    "Dep_Hour",
]

X = model_df[FINAL_FEATURES]
y = model_df["Delay_Class"]

print(f"Feature matrix shape: {X.shape}")
print(f"Class distribution:\n{y.value_counts(normalize=True).round(3)}")

# Cell 6 — Train Random Forest

The BTS dataset is significantly larger (~60 M rows over 10 years).
We stratify-sample **3 million rows** for training — triple the original —
which is still comfortably trainable on a Colab A100 in ~10 minutes.
Increase or decrease `SAMPLE_SIZE` as needed.

Class weights are computed dynamically from the actual distribution in
this dataset (rather than hardcoded 0.63 / 2.44).

In [ ]:
# ===== Cell 6: Train Random Forest =====
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

SAMPLE_SIZE = 3_000_000   # increase if RAM allows; 3 M is safe on 12 GB

sample_df = model_df.sample(n=min(SAMPLE_SIZE, len(model_df)), random_state=42)
X_s = sample_df[FINAL_FEATURES]
y_s = sample_df["Delay_Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X_s, y_s, test_size=0.2, random_state=42, stratify=y_s
)

# Recompute class weights from the sample (more accurate than hardcoding)
n0 = (y_s == 0).sum()
n1 = (y_s == 1).sum()
dynamic_weights = {0: len(y_s) / (2 * n0), 1: len(y_s) / (2 * n1)}
print("Dynamic class weights:", dynamic_weights)

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    min_samples_leaf=50,
    class_weight=dynamic_weights,
    n_jobs=-1,
    random_state=42,
)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
print(classification_report(y_test, y_pred, target_names=["On Time", "Delayed"]))

# Cell 7 — Recommender Function

Updated to accept the new `Dep_Hour` feature. If the user does not
specify an hour, we use the median departure hour observed at each
airport in the historical data — same pattern as before for weather.

In [ ]:
# ===== Cell 7: Recommender Function =====

def get_top5_recommendations(region, date_str, dep_hour=None):
    """
    Given a region name, a date string (YYYY-MM-DD), and an optional
    departure hour (0-23), return the top 5 airport+airline combos with
    the lowest predicted delay probability.

    dep_hour: if None, the median departure hour for each airport is used.
    """
    date = pd.to_datetime(date_str)

    # Temporal features from the input date
    year       = date.year
    month      = date.month
    dow        = date.dayofweek
    is_weekend = int(dow >= 5)
    is_holiday = int(any(abs((date - h).days) <= 2 for h in us_holidays))

    # Airports in the selected region
    regional_airports = airports_region[
        airports_region["Region"] == region
    ]["Dep_Airport"].tolist()

    if not regional_airports:
        print(f"No airports found for region: {region}")
        return None

    # All airport+airline combos that operated in this region
    combos = (
        model_df[model_df["Dep_Airport"].isin(regional_airports)]
        [["Dep_Airport", "Airline"]]
        .drop_duplicates()
        .reset_index(drop=True)
    )

    if combos.empty:
        print("No flight combinations found for this region.")
        return None

    # Median Dep_Hour per airport from historical data
    airport_medians = (
        model_df[model_df["Dep_Airport"].isin(regional_airports)]
        .groupby("Dep_Airport")[["Dep_Hour"]]
        .median()
        .reset_index()
    )
    combos = combos.merge(airport_medians, on="Dep_Airport", how="left")

    # Override with user-supplied hour if provided
    if dep_hour is not None:
        combos["Dep_Hour"] = int(dep_hour)

    # Add temporal features
    combos["Year"]      = year
    combos["Month"]     = month
    combos["DayOfWeek"] = dow
    combos["IsWeekend"] = is_weekend
    combos["IsHoliday"] = is_holiday

    # Encode using fitted label encoders (handle unseen labels)
    def safe_encode(encoder, values):
        known = set(encoder.classes_)
        return np.array([
            encoder.transform([v])[0] if v in known else -1
            for v in values
        ])

    combos["Airline_enc"]     = safe_encode(le_airline,  combos["Airline"])
    combos["Dep_Airport_enc"] = safe_encode(le_airport,  combos["Dep_Airport"])

    # Drop combos with unknown encodings
    combos = combos[
        (combos["Airline_enc"]     != -1) &
        (combos["Dep_Airport_enc"] != -1)
    ]

    # Predict delay probability
    X_pred = combos[FINAL_FEATURES].values
    delay_probs = rf.predict_proba(X_pred)[:, 1]

    combos["Delay_Probability"] = delay_probs
    combos["OnTime_Probability"] = 1 - delay_probs

    top5 = (
        combos[["Dep_Airport", "Airline", "Delay_Probability", "OnTime_Probability"]]
        .sort_values("Delay_Probability")
        .head(5)
        .reset_index(drop=True)
    )
    top5.index += 1
    top5["Delay_Probability"]  = top5["Delay_Probability"].map("{:.1%}".format)
    top5["OnTime_Probability"] = top5["OnTime_Probability"].map("{:.1%}".format)
    return top5

# Cell 8 — User Interaction

Same interface as the original. Optionally accepts a departure hour.

In [ ]:
# ===== Cell 8: User Interaction =====

VALID_REGIONS = list(region_to_states.keys())

print("=== Flight Delay Predictor (2014–2024 dataset) ===")
print(f"Available regions: {', '.join(VALID_REGIONS)}\n")

region_input   = input("Enter your departure region: ").strip().title()
date_input     = input("Enter your departure date (YYYY-MM-DD): ").strip()
hour_raw       = input("Enter preferred departure hour 0-23 (or press Enter to use median): ").strip()
hour_input     = int(hour_raw) if hour_raw.isdigit() else None

if region_input not in VALID_REGIONS:
    print(f"Invalid region. Please choose from: {', '.join(VALID_REGIONS)}")
else:
    try:
        results = get_top5_recommendations(region_input, date_input, dep_hour=hour_input)
        if results is not None:
            hour_str = f" at hour {hour_input}" if hour_input is not None else ""
            print(f"\nTop 5 airport+airline combinations for the {region_input}"
                  f" on {date_input}{hour_str}:\n")
            print(results.to_string())
    except Exception as e:
        print(f"Something went wrong: {e}")